## Deep_clip-like inspired approach for evaluation (BAnG)
### the original deep clip requires Python 2.7 + Theano 
https://deepclip.compbio.sdu.dk/
### Dataset used aptaDB   https://lmmd.ecust.edu.cn/aptadb/

In [14]:
import pandas as pd
import random
import requests
import time
from transformers import AutoTokenizer, AutoModel, EsmTokenizer, EsmModel
import torch
import torch.nn as nn
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoModelForMaskedLM
from torch.utils.data import Dataset, DataLoader
from pyaptamer.datasets import load_aptacom_x_y,load_li2014
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split


In [2]:
#blocked
!wget https://lmmd.ecust.edu.cn/aptadb/download/aptamer.csv
!wget https://lmmd.ecust.edu.cn/aptadb/download/interaction.csv
!wget https://lmmd.ecust.edu.cn/aptadb/download/protein.csv

aptamer_df = pd.read_csv("/kaggle/working/aptamer.csv",encoding="latin1")
interaction_df = pd.read_csv("/kaggle/working/interaction.csv",encoding="latin1")
protein_df = pd.read_csv("/kaggle/working/protein.csv",encoding="latin1")
#the positive samples are validated exprimentally , but the negative ones are made by switching the positions of proteins and aptamers indexes

--2026-06-03 20:02:28--  https://lmmd.ecust.edu.cn/aptadb/download/aptamer.csv
Resolving lmmd.ecust.edu.cn (lmmd.ecust.edu.cn)... 59.78.109.156
Connecting to lmmd.ecust.edu.cn (lmmd.ecust.edu.cn)|59.78.109.156|:443... connected.
HTTP request sent, awaiting response... 483 
2026-06-03 20:02:29 ERROR 483: (no description).

--2026-06-03 20:02:29--  https://lmmd.ecust.edu.cn/aptadb/download/interaction.csv
Resolving lmmd.ecust.edu.cn (lmmd.ecust.edu.cn)... 59.78.109.156
Connecting to lmmd.ecust.edu.cn (lmmd.ecust.edu.cn)|59.78.109.156|:443... connected.
HTTP request sent, awaiting response... 483 
2026-06-03 20:02:31 ERROR 483: (no description).

--2026-06-03 20:02:31--  https://lmmd.ecust.edu.cn/aptadb/download/protein.csv
Resolving lmmd.ecust.edu.cn (lmmd.ecust.edu.cn)... 59.78.109.156
Connecting to lmmd.ecust.edu.cn (lmmd.ecust.edu.cn)|59.78.109.156|:443... connected.
HTTP request sent, awaiting response... 483 
2026-06-03 20:02:32 ERROR 483: (no description).



In [3]:
!pip install pyaptamer==0.1.0a1


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 22.8 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 66.4 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 848.6/848.6 kB 33.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.8/163.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 271.6/271.6 kB 11.8 MB/s eta 0:00:00


In [33]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
# protein encoder : ems2 : we keep it frozen
prot_tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t33_650M_UR50D")
prot_encoder   = AutoModel.from_pretrained("facebook/esm2_t33_650M_UR50D").to(device)
for p in prot_encoder.parameters():
    p.requires_grad = False
prot_encoder.eval()
 
# nucleotide encoder ,nucTransfor same frozen 
nuc_tokenizer = AutoTokenizer.from_pretrained(
    "InstaDeepAI/nucleotide-transformer-500m-human-ref"
)
nuc_encoder = AutoModel.from_pretrained(
    "InstaDeepAI/nucleotide-transformer-500m-human-ref"
).to(device)
for p in nuc_encoder.parameters():
    p.requires_grad = False
nuc_encoder.eval()
 
NUC_DIM  = 1280   # nucTransfor hidden size   
MASK_TOKEN_ID = nuc_tokenizer.mask_token_id


Loading weights:   0%|          | 0/566 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: facebook/esm2_t33_650M_UR50D
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/390 [00:00<?, ?it/s]

EsmModel LOAD REPORT from: InstaDeepAI/nucleotide-transformer-500m-human-ref
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.decoder.weight      | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
esm.embeddings.position_ids | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


MASK token id: 2


In [22]:
x2,y2=load_li2014()
print(len(x2),len(y2))
print(x2.columns,y2.columns)
#print(y2)

2900 2900
Index(['aptamer', 'protein'], dtype='object') Index(['label'], dtype='object')


In [23]:
class AptaDataset(Dataset):
    def __init__(self, X, y):
        self.apt = list(X["aptamer"])
        self.prot = list(X["protein"])
        self.labels = torch.tensor(
            y["label"].values,
            dtype=torch.float32
        )

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        return self.apt[i], self.prot[i], self.labels[i]

label_map = {
    "negative": 0,
    "positive": 1
}

y2["label"] = y2["label"].map(label_map)

In [24]:
dataset=AptaDataset(x2,y2)

In [25]:
class ProjectionHead(nn.Module):
    def __init__(self, input_dim, proj_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, proj_dim),
            nn.ReLU(),
            nn.Linear(proj_dim, proj_dim)
        )

    def forward(self, x):
        return F.normalize(self.net(x), dim=-1)  # L2 normalize — required for cosine similarity


In [26]:
#change to max pooling 
def get_embedding(model, tokenizer, seqs, max_len=512, device="cpu"):
    """Mean-pool the last hidden state over non-padding tokens."""
    tokens = tokenizer(
        seqs, return_tensors="pt", padding=True,
        truncation=True, max_length=max_len
    ).to(device)
    with torch.no_grad():
        out = model(**tokens)
    mask = tokens["attention_mask"].unsqueeze(-1).float()
    return (out.last_hidden_state * mask).sum(1) / mask.sum(1)  # [B, hidden_dim]



In [27]:
def contrastive_loss(apt_z, prot_z, temperature=0.07):
    """
    Similarity matrix: every aptamer vs every protein in the batch.
    Diagonal = matching (positive) pairs.
    Cross entropy pulls diagonal to maximum.
    """
    logits = (apt_z @ prot_z.T) / temperature   # [B, B]
    labels = torch.arange(len(logits)).to(logits.device)
    loss_a = F.cross_entropy(logits,   labels)   # aptamer → protein direction
    loss_p = F.cross_entropy(logits.T, labels)   # protein → aptamer direction
    return (loss_a + loss_p) / 2


In [37]:
# ── 5. Setup ───────────────────────────────────────────────────────────────────
device = "cuda" if torch.cuda.is_available() else "cpu"

# projection heads — these are the only things being trained
apt_proj  = ProjectionHead(input_dim=1280, proj_dim=256).to(device)  # NT dim
prot_proj = ProjectionHead(input_dim=1280, proj_dim=256)  # match 650M dim

# encoders stay frozen
nuc_encoder.to(device)
nuc_encoder.eval()
prot_encoder.to(device)
prot_encoder.eval()
for p in nuc_encoder.parameters():  p.requires_grad = False
for p in prot_encoder.parameters(): p.requires_grad = False

optimizer = torch.optim.Adam(
    list(apt_proj.parameters()) + list(prot_proj.parameters()),
    lr=1e-4
)


In [38]:
X_train, X_test, y_train, y_test = train_test_split(
    x2,
    y2,
    test_size=0.2,
    random_state=42
)

train_loader = DataLoader(AptaDataset(X_train,y_train), batch_size=8, shuffle=True)
test_loader  = DataLoader(AptaDataset(X_test,y_test),  batch_size=8, shuffle=False)  # was missing



In [ ]:
print("Training CLIP projection heads...\n")

for epoch in range(50):
    apt_proj.train(); prot_proj.train()
    total_loss = 0

    for apt_seqs, prot_seqs, labels in train_loader:
        # get frozen embeddings
        apt_emb  = get_embedding(nuc_encoder,  nuc_tokenizer,  list(apt_seqs),  device=device)
        prot_emb = get_embedding(prot_encoder, prot_tokenizer, list(prot_seqs), device=device)

        # project to shared space
        apt_z  = apt_proj(apt_emb)
        prot_z = prot_proj(prot_emb)

        loss = contrastive_loss(apt_z, prot_z)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1:02d}/50 | Loss: {total_loss/len(train_loader):.4f}")


In [ ]:
print("\nEvaluating on test set\n")

apt_proj.eval(); prot_proj.eval()
all_scores, all_labels = [], []

with torch.no_grad():
    for apt_seqs, prot_seqs, labels in test_loader:
        apt_emb  = get_embedding(nuc_encoder,  nuc_tokenizer,  list(apt_seqs),  device=device)
        prot_emb = get_embedding(prot_encoder, prot_tokenizer, list(prot_seqs), device=device)

        apt_z  = apt_proj(apt_emb)
        prot_z = prot_proj(prot_emb)

        # cosine similarity as binding score (both are L2 normalized so dot product = cosine)
        scores = (apt_z * prot_z).sum(dim=-1)
        all_scores.extend(scores.cpu().numpy())
        all_labels.extend(labels.numpy())

preds = [1 if s > 0.5 else 0 for s in all_scores]
auc   = roc_auc_score(all_labels, all_scores)

print(f"AUC-ROC: {auc:.4f}")
